# Preparação única das cinco bases — T01

Este notebook usa os arquivos originais nas pastas dos grupos e publica um Excel tratado por grupo em `dados_tratados/`. Os modelos leem apenas esses Excel pelo carregador. Para atualizar os dados, regenere os mesmos cinco arquivos.

As lacunas permanecem explícitas: criar lags na grade completa, selecionar exemplos elegíveis depois, ajustar escala/encoding só no treino. O alvo real nunca é imputado.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preparacao_bases.py').is_file())
sys.path.insert(0, str(RAIZ))
from preparacao_bases import preparar_todas, carregar_base_tratada, PROTOCOLO


In [ ]:
destino = RAIZ / 'dados_tratados'
if not all((destino / f'base-tratada-{cfg["grupo"]}.xlsx').is_file() for cfg in PROTOCOLO.values()):
    preparar_todas(RAIZ)
print('Cinco bases tratadas em:', destino)
# Para regenerar após mudar fontes ou regras: preparar_todas(RAIZ, sobrescrever=True)


In [ ]:
resumo = []
for nome in PROTOCOLO:
    dados, meta = carregar_base_tratada(nome, RAIZ)
    bruto = meta['diagnostico_bruto']
    resumo.append({
        'grupo': meta['grupo'], 'base': nome, 'linhas': len(dados),
        'frequencia': meta['frequencia'], 'seed': meta['seed'],
        'treino': meta['linhas_treino'], 'teste': meta['linhas_teste'],
        'alvos_ausentes': int(dados[meta['alvo']].isna().sum()),
        'linhas_completas': int(dados.notna().all(axis=1).sum()),
        'nulos_fonte': bruto['linhas_com_nulos'],
        'duplicatas_fonte': bruto['duplicatas_exatas'],
        'datas_invalidas_fonte': bruto['datas_invalidas'],
        'frequencia_regular_fonte': bruto['frequencia_regular'],
    })
display(pd.DataFrame(resumo))

## Consumo nos modelos

O carregador devolve apenas medições e metadados; remove as colunas de controle do Excel para impedir que sejam usadas por engano como features. Use o corte temporal registrado mesmo depois de criar lags.

In [ ]:
df, meta = carregar_base_tratada('clima', RAIZ)
INICIO_TESTE = pd.Timestamp(meta['inicio_teste'])
# Após criar features causais, separar usando índice < ou >= INICIO_TESTE.
display(df.head())